# Stage 8 — Model Selection

The candidate models are regression models only. A dummy regressor is included as a baseline but is never selected as the final substantive model.

## 8.1 Prepare model inputs

In [704]:
X_train = train_model_data[kept_feature_columns]
y_train = train_model_data["log_u5mr_2023"]
X_test = test_model_data[kept_feature_columns]
y_test = test_model_data["log_u5mr_2023"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (147, 25)
X_test: (49, 25)


## 8.2 Define model pipelines

In [706]:
def scaled_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def tree_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model)
    ])

model_catalogue = {
    "Dummy mean": tree_pipeline(DummyRegressor(strategy="mean")),
    "Ridge": scaled_pipeline(Ridge(random_state=RANDOM_STATE)),
    "ElasticNet": scaled_pipeline(ElasticNet(max_iter=10000, random_state=RANDOM_STATE)),
    "KNN": scaled_pipeline(KNeighborsRegressor()),
    "Random Forest": tree_pipeline(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Extra Trees": tree_pipeline(ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": tree_pipeline(GradientBoostingRegressor(random_state=RANDOM_STATE))
}

list(model_catalogue.keys())

['Dummy mean',
 'Ridge',
 'ElasticNet',
 'KNN',
 'Random Forest',
 'Extra Trees',
 'Gradient Boosting']

# Stage 9 — Model Training and Cross-Validation

Cross-validation is performed on training countries only. Because the sample is small, CV results should be interpreted as approximate rather than definitive.

The country-level design produces fewer than 200 observations overall. With a 75/25 split, each five-fold CV validation fold contains only a small number of countries. This limitation is especially relevant for flexible ensemble models such as Random Forest and Extra Trees, which can fit complex patterns when the number of predictors is high relative to the sample size. The notebook therefore uses modest tuning, restricts the final predictor count, reports test-set bootstrap intervals, and avoids causal interpretation.

## 9.1 Cross-validation setup

In [709]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 9.2 Train and compare models using training CV

In [711]:
def evaluate_predictions(y_true_log, y_pred_log):
    return {
        "rmse_log": root_mean_squared_error(y_true_log, y_pred_log),
        "mae_log": mean_absolute_error(y_true_log, y_pred_log),
        "r2_log": r2_score(y_true_log, y_pred_log),
        "mae_original_scale": mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))
    }

cv_rows = []
fitted_initial_models = {}

for model_name, pipeline in model_catalogue.items():
    cv_scores = -cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )
    pipeline.fit(X_train, y_train)
    fitted_initial_models[model_name] = pipeline

    cv_rows.append({
        "model": model_name,
        "cv_rmse_log_mean": cv_scores.mean(),
        "cv_rmse_log_std": cv_scores.std()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_rmse_log_mean")
cv_results.to_csv(TABLE_DIR / "cv_model_comparison_train_only.csv", index=False)
display(cv_results)

best_cv_model_name = cv_results[cv_results["model"] != "Dummy mean"].iloc[0]["model"]
print("Best non-dummy model by training CV:", best_cv_model_name)

,model,cv_rmse_log_mean,cv_rmse_log_std
5,Extra Trees,0.371552,0.051505
4,Random Forest,0.376973,0.037556
6,Gradient Boosting,0.385421,0.048137
3,KNN,0.431822,0.036563
1,Ridge,0.452898,0.056710
2,ElasticNet,0.748254,0.045241
0,Dummy mean,1.083540,0.075684


Best non-dummy model by training CV: Extra Trees
